#### 1. Obtain protein IDs of orthologs per GENE from NCBI
#### 2. Using the IDS Fetch protein sequence ID from 'records' from the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) 
#### 3. Store nested dictionary of protein : sequences
#### 4. identify and extract unique sequences, human first then non-human
#### 5. Save sequences in FASTA format for each gene in /Fasta folder


#### 

In [2]:
run Kondrashov

* function: fetch protein record of each ortholog from NCBI

In [3]:
def fetch_gene_record(gene_id):
    """
    Fetch one Entrez Gene XML record and parse it with xmltodict.
    """
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="xml", retmode="text")
    record = x2d.parse(handle.read().decode("utf-8"))
    handle.close()
    return record

* function: extract protein ID from dictionary into a 'list'

In [4]:
def extract_protein_accessions(obj):
    """
    Recursively search the NCBI Gene XML dictionary and collect protein accessions.
    Returns accessions like XP_077799762.1 or NP_000148.2.
    """
    proteins = set()

    if isinstance(obj, dict):
        acc = obj.get("Gene-commentary_accession")
        ver = obj.get("Gene-commentary_version")

        if acc is not None:
            # Protein accessions usually start with NP_, XP_, or YP_
            if acc.startswith(("NP_", "XP_", "YP_")):
                if ver is not None:
                    proteins.add(f"{acc}.{ver}")
                else:
                    proteins.add(acc)

        for value in obj.values():
            proteins.update(extract_protein_accessions(value))

    elif isinstance(obj, list):
        for item in obj:
            proteins.update(extract_protein_accessions(item))

    return proteins


* RUN CODE USING DEFINED FUNCTIONS: obtain orthologs per gene 
* final output is 'records' contains gene, primate orthologs, details

In [ ]:
# obtain primate Orthologs per gene
result = {}

for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    search_record = Entrez.read(handle)
    handle.close()

    gene_ids = search_record["IdList"]
    print(f"{locus}: {search_record['Count']} -> {gene_ids[:4]}")


    result[locus] = {}

    for gene_id in gene_ids:
        try:
            gene_record = fetch_gene_record(gene_id)
            protein_ids = sorted(extract_protein_accessions(gene_record))
            result[locus][gene_id] = protein_ids
            print(f"  {gene_id}: {len(protein_ids)} proteins")

            # NCBI's request limit (10/sec with an API key)
            time.sleep(0.1)

        except Exception as expt:
            print(f"  Error with {locus} / {gene_id}: {expt}")
            result[locus][gene_id] = []

ABCD1: 35 -> ['215', '6367', '696794', '465923']
  215: 4 proteins
  6367: 5 proteins
  696794: 3 proteins
  465923: 1 proteins
  102120996: 5 proteins
  100409370: 1 proteins
  138379168: 3 proteins
  129475142: 4 proteins
  129023948: 2 proteins
  128576678: 2 proteins
  126945380: 1 proteins
  123628458: 2 proteins
  117078183: 1 proteins
  116812184: 2 proteins
  116541212: 1 proteins
  112615076: 2 proteins
  108535081: 2 proteins
  108294745: 1 proteins
  105867309: 2 proteins
  105824530: 2 proteins
  105707094: 1 proteins
  105598745: 1 proteins
  105555352: 2 proteins
  105521622: 2 proteins
  104661178: 2 proteins
  103251037: 1 proteins
  103232990: 3 proteins
  101125363: 1 proteins
  101029873: 1 proteins
  101015478: 1 proteins
  100974677: 1 proteins
  100947429: 3 proteins
  100586126: 1 proteins
  100441172: 2 proteins
  105467327: 2 proteins
ALPL: 35 -> ['249', '717809', '456600', '100401643']
  249: 8 proteins
  717809: 9 proteins
  456600: 6 proteins
  100401643: 10

* function: extract all protein ids per orthologs per genes into a list 

In [18]:
# create fasta folder
!mkdir fasta

def flatten_protein_ids(variant_dict):
    """
    Convert:
        {"variant1": [ids], "variant2": [ids]}
    into one unique list of protein IDs for that gene.
    """
    seq_ids = []
    seen = set()

    for variant_id, protein_ids in variant_dict.items():
        for pid in protein_ids:
            if pid not in seen:
                seq_ids.append(pid)
                seen.add(pid)

    return seq_ids



mkdir: fasta: File exists


* function: fetch protein details ('records') from NCBI
* modified

In [27]:
def fetch_protein_records(seq_ids):
    """
    Fetch GenBank protein records from NCBI.
    Uses chunks (at most 200) so the request does not become too large.
    """
    records = []
    chunk_size= 200
    for start in range(0, len(seq_ids), chunk_size):
        chunk = seq_ids[start:start + chunk_size]

        handle= Entrez.efetch(db="protein", rettype="gb", retmode="text", id=",".join(chunk))
        records.extend(list(SeqIO.parse(handle, "gb")))
        handle.close()

        time.sleep(0.35)

    return records

* function: get specie name from record eg 'Homo sapiens'

In [20]:
def get_species(record):
    """
    Retrieve species name from NCBI sequence record.

    Parameters
    ----------
    record : Bio.SeqRecord
        Sequence record.

    Returns
    -------
    str
        Species name.
    """
    description = record.description
    species = description.split(" [")[1][:-1]
    return species

* function; collect unique sequences; Homo sapiens first, then non Homo sapiens

In [21]:
def collect_unique_sequences_human_first(records):
    """
    Keep unique protein sequences.
    First collect unique Homo sapiens sequences,
    then collect unique non-human sequences.
    """
    seq_to_index = {}
    unique_records = []
    excluded_records = []

    inc = 0
    exc = 0

    # 1. Collect unique human sequences first
    for seq_record in records:
        sp = get_species(seq_record)
        if sp == "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}")
                inc += 1

    # 2. Collect other unique non-human sequences
    for seq_record in records:
        sp = get_species(seq_record)

        if sp != "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***")
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    return unique_records, excluded_records, inc, exc


* function; make a dict ('sequence_dict') comprising protein and its sequence

In [22]:
def make_sequence_dict(variant_dict, records):
    """
    Preserve your original nested structure, but replace each protein ID
    with its actual protein sequence.
    """
    #seq_by_id = {seq_record.id: str(seq_record.seq) for seq_record in records}
    seq_by_id = {}
    for seq_record in records:
        seq_by_id[seq_record.id: str(seq_record.seq)]= {}

    sequence_dict = {}

    for variant_id, protein_ids in variant_dict.items():
        sequence_dict[variant_id] = {}

        for pid in protein_ids:
            sequence_dict[variant_id][pid] = seq_by_id.get(pid, None)

    return sequence_dict

* RUN CODE USING CREATED FUNCTIONS
* Store dictionary containing protein id, unique sequences, etc 'summary_df' as dataframe
* save sequences in fasta file per gene

In [28]:
protein_sequences = {}
unique_records_by_gene = {}
excluded_records_by_gene = {}

summary = []

for gene, variant_dict in result.items():

    print("\n" + "=" * 80) #demarcation line
    print(datetime.now())
    print(f"{gene} orthologs")

    # Get all protein IDs for this gene from your result dictionary
    seq_ids = flatten_protein_ids(variant_dict)

    print(f"{gene}: {len(seq_ids)} protein IDs")
    print(f"Sequence IDs: {seq_ids[:10]}{' ...' if len(seq_ids) > 10 else ''}")

    if len(seq_ids) == 0:
        protein_sequences[gene] = {}
        unique_records_by_gene[gene] = []
        excluded_records_by_gene[gene] = []

        summary.append({
            "gene": gene,
            "n_protein_ids": 0,
            "n_records_fetched": 0,
            "n_unique_sequences": 0,
            "n_excluded_duplicates": 0,
            "fasta_file": None
        })

        continue

    # Fetch protein records from NCBI
    records = fetch_protein_records(seq_ids)

    print(f"{gene}: {len(records)} protein records fetched")

    # Store nested dictionary of actual sequences
    protein_sequences[gene] = make_sequence_dict(variant_dict, records)

    # Collect unique sequences, human first
    print(f"\n{gene} orthologs: unique primate sequences\n")

    unique_records, excluded_records, inc, exc = collect_unique_sequences_human_first(records)

    unique_records_by_gene[gene] = unique_records
    excluded_records_by_gene[gene] = excluded_records

    # Save FASTA file for that gene
    fasta_path = f"fasta/{gene}.fasta"

    with open(fasta_path, "w") as output:
        SeqIO.write(unique_records, output, "fasta")

    print(f"\nTotal: {inc} unique sequences, {exc} excluded")
    print(f"{fasta_path} saved!")

    summary.append({
        "gene": gene,
        "n_protein_ids": len(seq_ids),
        "n_records_fetched": len(records),
        "n_unique_sequences": inc,
        "n_excluded_duplicates": exc,
        "fasta_file": fasta_path
    })

summary_df = pd.DataFrame(summary)
summary_df


2026-06-23 16:37:24.135828
ABCD1 orthologs
ABCD1: 70 protein IDs
Sequence IDs: ['NP_000024.2', 'NP_001427676.1', 'XP_047297873.1', 'XP_054182656.1', 'NP_002981.2', 'XP_047290405.1', 'XP_047290406.1', 'XP_054169574.1', 'XP_054169575.1', 'XP_001085640.2'] ...
ABCD1: 70 protein records fetched

ABCD1 orthologs: unique primate sequences

0:	NP_000024.2	(745 aa)	ATP-binding cassette sub-family D member 1 isoform 1 [Homo sapiens]
1:	NP_001427676.1	(845 aa)	ATP-binding cassette sub-family D member 1 isoform 2 [Homo sapiens]
2:	XP_047297873.1	(575 aa)	ATP-binding cassette sub-family D member 1 isoform X2 [Homo sapiens]
	XP_054182656.1	(575 aa)	ATP-binding cassette sub-family D member 1 isoform X2 [Homo sapiens]	*** same as sequence 2 excluded ***
3:	NP_002981.2	(93 aa)	C-C motif chemokine 22 precursor [Homo sapiens]
4:	XP_047290405.1	(106 aa)	C-C motif chemokine 22 isoform X1 [Homo sapiens]
	XP_047290406.1	(93 aa)	C-C motif chemokine 22 isoform X2 [Homo sapiens]	*** same as sequence 3 exclude

,gene,n_protein_ids,n_records_fetched,n_unique_sequences,n_excluded_duplicates,fasta_file
0,ABCD1,70,70,62,8,fasta/ABCD1.fasta
1,ALPL,169,169,71,98,fasta/ALPL.fasta
2,AR,73,73,71,2,fasta/AR.fasta
3,ATP7B,266,266,209,57,fasta/ATP7B.fasta
4,BTK,79,79,40,39,fasta/BTK.fasta
5,CASR,87,87,50,37,fasta/CASR.fasta
6,CBS,255,255,103,152,fasta/CBS.fasta
7,CFTR,56,56,51,5,fasta/CFTR.fasta
8,CYBB,51,51,34,17,fasta/CYBB.fasta
9,F7,87,87,81,6,fasta/F7.fasta
